# ***Cell 1 Imports & Output Directory***

In [ ]:
import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

OUTPUT_DIR = "/content/Week_2_Package"
VIS_DIR = os.path.join(OUTPUT_DIR, "visualizations")

os.makedirs(VIS_DIR, exist_ok=True)

# ***Cell 2 Fetch Dataset from Drive***

In [ ]:
from google.colab import drive
import glob, zipfile

drive.mount("/content/drive")

zip_matches = glob.glob("/content/drive/MyDrive/**/cleaned_dataset.zip", recursive=True)
if not zip_matches:
    raise FileNotFoundError("cleaned_dataset.zip not found in Drive")

with zipfile.ZipFile(zip_matches[0], "r") as zf:
    zf.extractall("/content")

print(f"Extracted: {zip_matches[0]}")

# ***Cell 3 Load metadata.csv***

In [ ]:
meta_files = []

for root, dirs, files_ in os.walk("/content"):
    for file in files_:
        if file.lower() == "metadata.csv":
            meta_files.append(os.path.join(root, file))

print("Found metadata files:")
for path in meta_files:
    print(path)

In [ ]:
meta_path = "/content/cleaned_dataset/metadata.csv"
df = pd.read_csv(meta_path)

print("Metadata path:", meta_path)
print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

df.head()

# ***Cell 4 Capacity Cleaning & Discharge Data***

In [ ]:
df["Capacity"] = pd.to_numeric(
    df["Capacity"].astype(str).str.replace(r"[\[\]]", "", regex=True),
    errors="coerce"
)

dis_df = df[
    (df["type"] == "discharge") &
    (df["Capacity"] > 0.05)
].copy()

print(f"Discharge samples: {len(dis_df)}")
print(f"Batteries: {dis_df['battery_id'].nunique()}")

# ***Cell 5 Initial Capacity & SOH Calculation***

In [ ]:
c0 = dis_df.groupby("battery_id")["Capacity"].first()

dis_df["initial_capacity"] = dis_df["battery_id"].map(c0)
dis_df["SOH"] = dis_df["Capacity"] / dis_df["initial_capacity"]

dis_df[["battery_id", "test_id", "Capacity", "initial_capacity", "SOH"]].head()

# ***Cell 6 Basic SOH Statistics***

In [ ]:
print("SOH Statistics")
print(dis_df["SOH"].describe())

print("\nSOH by Battery")
print(
    dis_df.groupby("battery_id")["SOH"]
    .agg(["min", "max", "mean", "count"])
)

# ***Cell 7 Test-Type Distribution by Cell***

In [ ]:
plt.style.use(
    "seaborn-v0_8-whitegrid"
    if "seaborn-v0_8-whitegrid" in plt.style.available
    else "default"
)

type_counts = df.groupby(["battery_id", "type"]).size().unstack(fill_value=0)
type_counts = type_counts.loc[sorted(type_counts.index)]

fig, ax = plt.subplots(figsize=(10, 6))
type_counts.plot(kind="bar", stacked=False, ax=ax, colormap="tab10", edgecolor="black", linewidth=0.5)

ax.set_title("Test-Type Distribution by Battery Cell", fontsize=14, fontweight="bold")
ax.set_xlabel("Battery Cell", fontsize=11)
ax.set_ylabel("Number of Records", fontsize=11)
ax.legend(title="Test Type", fontsize=9, title_fontsize=10, loc="upper left", bbox_to_anchor=(1.01, 1.0))
ax.tick_params(axis="x", rotation=0)

for container in ax.containers:
    ax.bar_label(container, fontsize=8, padding=2)

plt.tight_layout()
plt.savefig(os.path.join(VIS_DIR, "celltype_distribution.png"), dpi=300, bbox_inches="tight")
plt.show()
plt.close()

# ***Cell 8 Degradation Trajectory Visualization***

In [ ]:
plt.style.use(
    "seaborn-v0_8-whitegrid"
    if "seaborn-v0_8-whitegrid" in plt.style.available
    else "default"
)

battery_ids = sorted(dis_df["battery_id"].unique())
n_cells = len(battery_ids)
n_cols = min(n_cells, 4)
n_rows = int(np.ceil(n_cells / n_cols))

palette = sns.color_palette("tab10", n_colors=n_cells)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.2 * n_cols, 3.6 * n_rows + 0.6), sharey=True)
axes = np.atleast_1d(axes).flatten()

for ax, bid, color in zip(axes, battery_ids, palette):
    cell_data = dis_df[dis_df["battery_id"] == bid].sort_values("test_id")
    ax.plot(cell_data["test_id"], cell_data["SOH"], color=color, linewidth=1.4, alpha=0.85)
    ax.scatter(cell_data["test_id"], cell_data["SOH"], color=color, s=10, alpha=0.5)
    ax.axhline(0.70, color="red", linestyle="--", linewidth=1, alpha=0.7)
    ax.set_title(f"Cell {bid}", fontsize=11, fontweight="bold")
    ax.set_xlabel("Cycle Index", fontsize=9)
    ax.tick_params(labelsize=8)
    ax.grid(alpha=0.3)

axes[0].set_ylabel("State of Health (SOH)", fontsize=9)

for ax in axes[n_cells:]:
    ax.set_visible(False)

handles = [plt.Line2D([0], [0], color="red", linestyle="--", label="EOL Threshold (70% SOH)")]
fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, -0.02), ncol=1, fontsize=9, frameon=False)
fig.suptitle("Degradation Trajectory by Battery Cell", fontsize=14, fontweight="bold", y=1.02)

plt.tight_layout(rect=[0, 0.02, 1, 0.96])
plt.savefig(os.path.join(VIS_DIR, "sequence_degradation.png"), dpi=300, bbox_inches="tight")
plt.show()
plt.close()

# ***Cell 9 Degradation Stage Distribution by Cell***

In [ ]:
bins = [-np.inf, 0.70, 0.85, np.inf]
labels = [
    "Accelerated Aging (<70%)",
    "Knee Transition (70-85%)",
    "Healthy (>85%)"
]
dis_df["stage"] = pd.cut(dis_df["SOH"], bins=bins, labels=labels)

stage_by_cell = dis_df.groupby(["battery_id", "stage"], observed=False).size().unstack(fill_value=0)
stage_by_cell = stage_by_cell.reindex(columns=labels, fill_value=0)
stage_by_cell = stage_by_cell.loc[sorted(stage_by_cell.index)]
stage_pct = stage_by_cell.div(stage_by_cell.sum(axis=1), axis=0) * 100

colors = ["#d62728", "#ff7f0e", "#2ca02c"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

stage_by_cell.plot(kind="bar", stacked=True, ax=axes[0], color=colors, edgecolor="black", linewidth=0.5)
axes[0].set_title("Degradation Stage Counts by Cell", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Battery Cell")
axes[0].set_ylabel("Number of Cycles")
axes[0].tick_params(axis="x", rotation=0)
axes[0].legend(title="Stage", fontsize=8, title_fontsize=9)

stage_pct.plot(kind="bar", stacked=True, ax=axes[1], color=colors, edgecolor="black", linewidth=0.5)
axes[1].set_title("Degradation Stage Share by Cell (%)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Battery Cell")
axes[1].set_ylabel("Share of Cycles (%)")
axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(title="Stage", fontsize=8, title_fontsize=9)
axes[1].set_ylim(0, 100)

fig.suptitle("Degradation Stage Distribution --- Cell-wise Breakdown", fontsize=14, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig(os.path.join(VIS_DIR, "degradation_stages.png"), dpi=300, bbox_inches="tight")
plt.show()
plt.close()

# ***Cell 10 Check Generated Package***

In [ ]:
for root, dirs, files_ in os.walk(OUTPUT_DIR):
    level = root.replace(OUTPUT_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files_:
        print(f"{indent}  {file}")

# ***Cell 11 Create ZIP***

In [ ]:
zip_path = shutil.make_archive(
    "/content/Week_2_Package",
    "zip",
    OUTPUT_DIR
)

print(f"Package created: {zip_path}")

# ***Cell 12 Download***

In [ ]:
files.download("/content/Week_2_Package.zip")